# 09 — Own Model Export & Central Model Registry
### AI Interview System — 100% Project-Owned ML Pipeline
This notebook finalizes production deployment:
1. Verifies promotion gate approval status from `reports/fine_tuned_model_evaluation.json`.
2. Packages semantic version `ai-interview-question-generator-v1.0.0` with full architecture metadata.
3. Registers the model in `models/model_registry.json`.
4. Promotes model status to `production` and verifies offline inference.


In [ ]:
# Cell 1: Verify Gate Approval & Register Model
import os
import json
from pathlib import Path
from datetime import datetime, timezone
from model_registry import registry

WORKSPACE_DIR = Path(os.getcwd())
with open(WORKSPACE_DIR / "reports" / "fine_tuned_model_evaluation.json", "r", encoding="utf-8") as f:
    eval_report = json.load(f)

assert eval_report["promotion_gate"]["promotion_status"] == "approved", "Model failed promotion gate! Cannot deploy to production."

model_rec = {
    "model_id": "ai-interview-question-generator-v1.0.0",
    "model_name": "AI Interview Own Transformer Question Generator",
    "version": "1.0.0",
    "capability": "question_generator",
    "model_type": "scratch_trained",
    "storage_path": "models/interview_model",
    "parameters": "14.2M (Project Architecture)",
    "metrics": eval_report["specialized_model_metrics"],
    "status": "production",
    "created_at": datetime.now(timezone.utc).isoformat()
}

registry.register_model(model_rec)
registry.set_active_model("question_generator", "ai-interview-question-generator-v1.0.0")

print("Successfully registered and activated model in production:")
print(json.dumps(model_rec, indent=2))


In [ ]:
# Cell 2: Verify Offline Inference
from agents.question_generator_agent import QuestionGeneratorAgent

agent = QuestionGeneratorAgent()
profile = {"role": "Backend Engineer", "experience_level": "Senior"}
result = agent.generate_question(profile, target_topic="System Design", difficulty_level="Advanced")

print("=== LIVE INFERENCE VERIFICATION ===")
print("Generated Question:", result["interview_question"])
print("Model Metadata:", json.dumps(result["model_metadata"], indent=2))
print("Stage 09 Completed Successfully. Pipeline 100% Ready.")
